## MULTIVARIADA

In [2]:
import pandas as pd
import numpy as np

raw_dataset = pd.read_excel('25-testcar-2026-01-21.xlsx') 
preprocessed_dataset = raw_dataset.copy()

print("Dimensiones iniciales del dataset = [filas, columnas]:", raw_dataset.shape)


pd.options.display.max_columns = None
raw_dataset.head()


columnas_borrar = [ 
    "Model Year",
    "FE_UNIT",
]

preprocessed_dataset.drop(columns=columnas_borrar, inplace=True, errors='ignore')


Dimensiones iniciales del dataset = [filas, columnas]: (4287, 67)


In [3]:

columnas_borrar = [ 
    #conservando el codigo (legibilidad)
    'Drive System Description',
    'Transmission Overdrive Desc',
    'Tested Transmission Type', 
    
    #conservando la descripcion (legibilidad)
    'Shift Indicator Light Use Cd', 
    'Test Procedure Cd', 
    'Test Fuel Type Cd',
    'Aftertreatment Device Cd',
    'Averaging Method Cd'
]

preprocessed_dataset.drop(columns=columnas_borrar, inplace=True, errors='ignore')

print("Dimensiones del dataset luego de la limpieza = ", preprocessed_dataset.shape)

Dimensiones del dataset luego de la limpieza =  (4287, 57)


In [10]:
cuantil_1 = preprocessed_dataset['RND_ADJ_FE'].quantile(0.25)
cuantil_3 = preprocessed_dataset['RND_ADJ_FE'].quantile(0.75)

iqr = cuantil_3 - cuantil_1

limite_inferior = cuantil_1 - 1.5 * iqr
limite_superior = cuantil_3 + 1.5 * iqr

mpg_sin_outliers = preprocessed_dataset[(preprocessed_dataset['RND_ADJ_FE'] >= limite_inferior) & (preprocessed_dataset['RND_ADJ_FE'] <= limite_superior)]

print(f"El umbral superior estadístico calculado es: {limite_superior:.2f} MPG")
print(mpg_sin_outliers['RND_ADJ_FE'].describe())

El umbral superior estadístico calculado es: 72.80 MPG
count    3738.000000
mean       32.635045
std        11.424164
min         0.000000
25%        23.800000
50%        31.600000
75%        39.800000
max        72.700000
Name: RND_ADJ_FE, dtype: float64


Agrupación (Groupby): Agrupar el dataset por Vehicle Manufacturer Name y calcular el promedio (media) de sus variables objetivo: RND_ADJ_FE (consumo) y CO2 (g/mi) (emisiones).

In [ ]:
fabricantes = mpg_sin_outliers.groupby('Vehicle Manufacturer Name')[['RND_ADJ_FE', 'CO2 (g/mi)']].mean().reset_index()
#agregar columna si es elecrico/ hidrogeno
fabricantes['Tipo de Vehículo'] = np.where(fabricantes['typ'] > 100, 'Eléctrico/Hidrógeno', 'Combustión')    

In [20]:
top_eficientes = fabricantes.sort_values(by='RND_ADJ_FE', ascending=False).head(5)
top_contaminantes = fabricantes.sort_values(by='RND_ADJ_FE', ascending=True).head(5)



In [21]:

print("--- TOP 5 Fabricantes Más Eficientes ---")
print(top_eficientes)

print("\n--- TOP 5 Fabricantes Más Contaminantes ---")
print(top_contaminantes)

--- TOP 5 Fabricantes Más Eficientes ---
   Vehicle Manufacturer Name  RND_ADJ_FE  CO2 (g/mi) Tipo de Vehículo
25       Vinfast Trading and   52.475000         NaN       Combustión
1          Bugatti Rimac LLC   44.950000         NaN       Combustión
14                     MAZDA   40.401250  228.592724       Combustión
24                    Toyota   40.350429  238.065567       Combustión
7                      Honda   39.604286  240.580912       Combustión

--- TOP 5 Fabricantes Más Contaminantes ---
   Vehicle Manufacturer Name  RND_ADJ_FE  CO2 (g/mi) Tipo de Vehículo
2                    CZV Inc   16.000000  560.915000       Combustión
9        Ineos Automotive Li   18.000000  510.838333       Combustión
21               Rolls-Royce   20.400000  466.368990       Combustión
5                    Ferrari   21.447059  440.303972       Combustión
29              aston martin   23.862500  399.534375       Combustión


Construcción de Rankings de Impacto: Se identificarán de forma explícita los Top 5 fabricantes más eficientes y los Top 5 fabricantes más contaminantes, permitiendo un contraste directo de los extremos del mercado.

Validación Visual y Estadística: Se representarán los resultados mediante un diagrama de dispersión (scatter plot) bidimensional cruzando MPG y emisiones de $CO_2$ por marca, respaldado por una matriz de correlación para evaluar la magnitud y dirección de la relación conjunta.